# Session 4 — From chat to API\n\nWe run this together. By the end you'll have made programmatic model calls, seen temperature with your own eyes, produced structured output into a table, and batched a real classification. **None of this requires writing code today** — run the cells, read what they do, change small things when invited.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path("..").resolve()))   # notebooks/ -> repo root
import course_utils

try:
    client = course_utils.get_client(verbose=True)
except RuntimeError:
    # First time here without check_setup? We can store the key now too:
    import getpass, os
    key = getpass.getpass("Paste your API key (input stays hidden): ")
    course_utils._persist("GEMINI_API_KEY", key)
    os.environ["GEMINI_API_KEY"] = key
    client = course_utils.get_client(verbose=True)

MODEL = course_utils.MODEL
print("Client ready.")

## 1. Your first programmatic call\n\nSame model as the chat window — minus the app around it. Note the three things we control: the **model**, the **contents**, the **config**.

In [ ]:
r = client.models.generate_content(
    model=MODEL,
    contents="In one sentence: why might a researcher prefer an API over a chat window?",
)
print(r.text)

## 2. Temperature — the randomness dial, demonstrated\n\nSame prompt, five runs, twice: once at temperature 1 (default-ish), once at 0. Watch what happens to the variation. **This is the answer to Assignment 1, Exercise 2** — the instability you measured has a dial.

In [ ]:
prompt = "Classify this app review's sentiment as exactly one word, positive/negative/mixed: 'Great features when it works, which is rarely.'"
for temp in [1.0, 0.0]:
    answers = []
    for _ in range(5):
        r = client.models.generate_content(model=MODEL, contents=prompt, config={"temperature": temp})
        answers.append(r.text.strip())
    print(f"temperature {temp}: {answers}")

## 3. Structured output — answers as data, not prose\n\nWe ask for JSON and force it with `response_mime_type`. The result drops straight into a table. This is the pattern behind every pipeline in this course.

In [ ]:
import json, pandas as pd
abstract = open("../a1/abstracts.md", encoding="utf-8").read().split("## Abstract 01")[1].split("## Abstract 02")[0]
r = client.models.generate_content(
    model=MODEL,
    contents=f"""Extract from the abstract below. Verbatim quote for key_finding.
Any field not stated -> the string NOT REPORTED.
Return JSON with keys: research_question, method, sample, key_finding.
ABSTRACT: {abstract}""",
    config={"temperature": 0, "response_mime_type": "application/json"},
)
row = json.loads(r.text)
pd.DataFrame([row])

## 4. Batching — many items per call\n\nRate limits count *requests*, so we send 10 texts in one request and get 10 labeled rows back. (On a free-tier key this pattern is what keeps you comfortable; on Vertex credit it's still just faster.)

In [ ]:
import pandas as pd, json
sample = pd.read_csv("../data/sample_200.csv").head(10)
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(sample.concern_text))
codebook = open("../data/codebook_concerns.md", encoding="utf-8").read()
r = client.models.generate_content(
    model=MODEL,
    contents=f"""Using ONLY the codebook below, assign one category (JOB/SURV/SKILL/FAIR/REL) to each numbered text.
Return a JSON list of objects with keys: n, category.
CODEBOOK:\n{codebook}\n\nTEXTS:\n{numbered}""",
    config={"temperature": 0, "response_mime_type": "application/json"},
)
labels = pd.DataFrame(json.loads(r.text))
sample.reset_index(drop=True).join(labels.set_index(labels.n - 1))[["resp_id", "concern_text", "category"]]

## 5. What a call costs — the mental model\n\nAPIs charge per **token** (~3/4 of a word). Count before you run, and numbers never surprise you.

In [ ]:
toks = client.models.count_tokens(model=MODEL, contents=numbered)
print(f"Those 10 texts = {toks.total_tokens} input tokens.")
print(f"All 1,200 survey rows, extrapolated: ~{toks.total_tokens * 120:,} tokens -> free on Flash's free tier; cents on paid.")

**Take-home experiment (10 min, optional):** change the batch in section 4 to rows 11–20 and rerun. Then look at one label you disagree with — you've just previewed Assignment 3's verification step V3.